In [ ]:
from cosipy.nonimaging.bgo.bc_tools_localization import BGOLocalizerBCT
import numpy as np


In [ ]:
# Inizializza con i tre LUT (pickle) e nside

dir_path = "/data/test_newrepo/"
run_name="run10"
localizer = BGOLocalizerBCT(
    soft_loctable_path=dir_path+'/soft_lut_' + run_name + '.pkl',
    medium_loctable_path=dir_path+'/medium_lut_' + run_name + '.pkl',
    hard_loctable_path= dir_path+'/hard_lut_' + run_name + '.pkl',
    nside=64,
)

In [ ]:
def ra_dec_to_theta_phi(ra, dec):

    theta = 90-dec
    
    phi = ra
    
    return theta, phi
def spherical_to_radec_deg(theta_deg, phi_deg):

    dec = 90.0 - theta_deg
    ra = phi_deg % 360.0
    return ra, dec


In [ ]:
# Counts order ['BGO_X0','BGO_X1','BGO_Y0','BGO_Y1','BGO_Z0','BGO_Z1']
# Simulated GRB
# Spectrum Band 10 10000 -1.9 -3.7 230
# Flux 14.58 ph/cm2/s
# True position theta=84.021 phi=49.922

true_ra,true_dec = spherical_to_radec_deg(84.021,49.922)
# s_counts = [46, 316, 33, 374, 47, 34]

# #for testing purpose generate random Poissonian background 
# mean_counts = np.array([57.6053, 58.7157, 51.4131, 48.2891, 47.7293, 45.9617])
# random_bkg = np.random.poisson(np.array([mean_counts[3],mean_counts[2],mean_counts[5],mean_counts[4],mean_counts[1],mean_counts[0]])*20)

# b_counts = np.array([mean_counts[3],mean_counts[2],mean_counts[5],mean_counts[4],mean_counts[1],mean_counts[0]])*20
# s_counts = s_counts+random_bkg

s_counts = [2356.00,2431.00,3116.00,2863.00,1924.00,1951.00]
b_counts = [1079.35,1131.83,1277.07,1323.91,1098.75,1089.84]
# z1: signal=1951.00, background=1089.84, net=861.16
# z0: signal=1924.00, background=1098.75, net=825.25
# x1: signal=2431.00, background=1131.83, net=1299.17
# x0: signal=2356.00, background=1079.35, net=1276.65
# y1: signal=2863.00, background=1323.91, net=1539.09
# y0: signal=3116.00, background=1277.07, net=1838.93

In [ ]:

from astropy.coordinates import SkyCoord
import astropy.units as u
from scoords import Attitude, SpacecraftFrame
ori_file  = "/home/cosi/cosi/data/background/dc4/DC3_final_530km_3_month_with_slew_15sbins_GalacticEarth_SAA.ori"

data = np.loadtxt(ori_file, usecols=(1, 2, 3, 4, 5, 6, 7, 8), delimiter=' ', skiprows=1, comments=("#", "EN"))

In [ ]:
# esempio: valore target
time_grb = 1835517732.9649906

# colonna del tempo (prima colonna di data)
tempi = data[:, 0]

# indice del valore più vicino
idx = np.argmin(np.abs(tempi - time_grb))

# riga corrispondente
riga_piu_vicina = data[idx]

print("Indice:", idx)
print("Tempo più vicino:", tempi[idx])
print("Riga:", riga_piu_vicina)

i = idx+1

In [ ]:

x_pointing = SkyCoord(data[:, 2][i]*u.deg, data[:, 1][i]*u.deg, frame='galactic')
z_pointing = SkyCoord(data[:, 4][i]*u.deg, data[:, 3][i]*u.deg, frame='galactic')
#x_pointing = SkyCoord(0*u.deg, 0*u.deg, frame='galactic')
#z_pointing = SkyCoord(0*u.deg, 90*u.deg, frame='galactic')
attitude = Attitude.from_axes(x=x_pointing, z=z_pointing, frame='galactic')

result = localizer.localize(s_counts, b_counts,attitude=attitude)
print(
    result["label"],
    result["sqrt_ts"],
    result["ra_deg"],
    result["dec_deg"],
    result["eq_radius_deg"],
)

In [ ]:
result

In [ ]:
ra_dec_to_theta_phi(result["ra_deg"],result["dec_deg"])

In [ ]:
# Plot opzionale
%matplotlib inline

from astropy.coordinates import SkyCoord
import astropy.units as u

true_coord = SkyCoord(ra=true_ra * u.deg, dec=true_dec * u.deg, frame="icrs")
localizer.plot(result,true_coord=true_coord, show=True,save_path="/tmp/plot.png")
